# BFS & DFS Pattern Playbook - Trees
**Topic:** Trees | **Type:** Pattern notebook (many problems, few templates)

This notebook is the direct follow-on to **section 2.5** of `0. tree_traversals_dfs_and_bfs.ipynb`
("when to reach for BFS vs. DFS"). There we established the *decision rule*. Here we prove the payoff:
once you own **one baseline template per pattern**, a good slice of the tree-shaped Blind 75 / NeetCode 150
problems stops being "several different algorithms to memorise" and becomes "1 template, plugged in
several different ways" - the same relationship section **2.2 -> 2.3** demonstrated for level-order BFS
(own the level-size snapshot once, then right-side-view / zigzag / bottom-up are each a one-line edit).

> **Scope note:** this notebook stays strictly on **trees** (every problem's input is a `TreeNode`-based
> binary tree, possibly with one small augmentation like parent pointers). Grid/adjacency-list graph
> problems (Number of Islands, Course Schedule, Word Ladder, Clone Graph, etc.) belong to the *same*
> templates conceptually, but are deliberately left for a separate graph-focused notebook later.

**How to read this notebook:** every variant cell starts with **"Same as baseline, except:"** - read that
line first. If you can already predict the diff before reading the code, you've internalised the pattern.

- **Part A - BFS:** one generic function, `bfs_generic`, solves **2** tree problems by varying its
  3 arguments - including one that shows BFS isn't limited to "downward" traversal.
- **Part B - DFS:** two templates - a *return-a-value* recursion (**4** problems) and a *path
  backtracking* recursion (**4** problems).


---
# Part A - BFS: One Generic Template

**The insight:** every BFS problem is really the same question - *"starting from some set of nodes, spread
outward one edge at a time, and either (a) stop the instant you meet a target, or (b) record how far every
node is from its nearest starting point."* On a tree, "one edge" normally means "one parent-to-child hop" -
but as A.2 below shows, it doesn't have to.

So instead of writing BFS twice from scratch, we write it **once**, parameterised by three things:

| Parameter | Answers |
|---|---|
| `sources` | Where does the ripple start? (usually `[root]`, or any single node) |
| `neighbors_fn(node)` | What counts as one step away from a node? |
| `is_target_fn(node)` (optional) | Should we stop early the instant we find a match? |


In [ ]:
from collections import deque

def bfs_generic(sources, neighbors_fn, is_target_fn=None):
    """One BFS to rule them all.

    - sources: iterable of starting nodes (single-source BFS = [start]).
    - neighbors_fn(node) -> iterable of nodes one step away from `node`.
    - is_target_fn(node) -> bool, optional early-stop condition.

    Returns:
      - if is_target_fn is given: the number of EDGES from the nearest source to the
        first node that satisfies it, or -1 if no reachable node ever does.
      - otherwise: a dict {node: distance_from_nearest_source} for every reached node.
    """
    dist = {s: 0 for s in sources}         # every source starts at distance 0
    q = deque(sources)
    while q:
        node = q.popleft()
        if is_target_fn is not None and is_target_fn(node):
            return dist[node]              # early stop - guaranteed to be the NEAREST match
        for nxt in neighbors_fn(node):
            if nxt not in dist:            # dist doubles as the visited set
                dist[nxt] = dist[node] + 1
                q.append(nxt)
    return -1 if is_target_fn is not None else dist


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val; self.left = left; self.right = right

def build_tree(values):
    """Level-order list -> tree, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values):
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def find_node(root, val):
    """Test helper only - LeetCode hands you the node directly, not a value to search for."""
    if not root: return None
    if root.val == val: return root
    return find_node(root.left, val) or find_node(root.right, val)


## A.1 - LC 111: Minimum Depth of Binary Tree

**Same as baseline, except:** `sources = [root]` (single source), `neighbors_fn` returns the existing
children of a tree node, and `is_target_fn` checks "is this a leaf?" - this is exactly the worked example
from notebook `0.`, sections 2.4/2.5, now expressed as one call to `bfs_generic`.

Note `bfs_generic` counts **edges**, but LC 111's "depth" counts the root itself as depth 1 - so the
answer is `edges + 1`.


In [ ]:
def min_depth(root):
    if not root:
        return 0
    edges = bfs_generic(
        [root],
        lambda n: [c for c in (n.left, n.right) if c],   # neighbours = existing children
        lambda n: not n.left and not n.right,             # target = a leaf
    )
    return edges + 1


t = build_tree([1, 2, 3, 4, 5, None, None, None, None, None, 6])   # the tree from notebook 0's example
print("min depth:", min_depth(t))          # expect 2 (leaf `3` sits one edge from the root)
print("empty tree:", min_depth(None))      # expect 0
print("single node:", min_depth(build_tree([1])))   # expect 1


## A.2 - LC 863: All Nodes Distance K in Binary Tree

**Same as baseline, except:** BFS on a tree normally only ever moves *downward* (parent -> child), because
that's the only edge a `TreeNode` gives you directly. This problem asks "which nodes are exactly `k` hops
from some target node?" - and the answer can be **above** the target, or in a completely different branch.
Neither is reachable by only following `.left`/`.right`.

The fix doesn't touch `bfs_generic` at all - it only changes what `neighbors_fn` considers a "neighbour."
One pre-pass builds a `child -> parent` map; `neighbors_fn` then offers **three** directions instead of two
(`left`, `right`, *and* `parent`), which turns the tree into an undirected structure for the purposes of
this one BFS. `sources = [target]`, no early stop (we want *every* node at distance `k`, not the first one),
so we take the full distance dict and filter it.


In [ ]:
def build_parent_map(root):
    """One traversal: map every child node to its parent (the root has no entry)."""
    parent = {}
    stack = [root]
    while stack:
        node = stack.pop()
        if node.left:
            parent[node.left] = node; stack.append(node.left)
        if node.right:
            parent[node.right] = node; stack.append(node.right)
    return parent


def distance_k(root, target, k):
    parent = build_parent_map(root)

    def neighbors(node):                    # THREE directions now, not two - this is the whole change
        for n in (node.left, node.right, parent.get(node)):
            if n:
                yield n

    dist = bfs_generic([target], neighbors)   # no is_target_fn -> we want every node, not the first match
    return [node.val for node, d in dist.items() if d == k]


tk = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
target_node = find_node(tk, 5)
print(sorted(distance_k(tk, target_node, 2)))   # expect [1, 4, 7]


## A.3 - Recap

| Problem | `sources` | `neighbors_fn` | `is_target_fn` | Answer shape |
|---|---|---|---|---|
| 111 Min Depth | `[root]` | children only (downward) | is a leaf | `edges + 1` |
| 863 Distance K | `[target]` | children **+ parent** (any direction) | *(none)* | filter `dist == k` |

The lesson in A.2 isn't really about BFS - it's that **a tree is just a graph with an extra rule ("no
cycles, one parent")**, and the moment a question needs to move in a direction that rule doesn't give you
for free (upward), you hand-build the missing edge (a parent map) and the exact same `bfs_generic` still
works. That's also the natural bridge into general graph BFS, which the next notebook picks up.


---
# Part B - DFS: Two Templates for Two Kinds of Questions

DFS doesn't collapse into one parameterised function the way BFS does here, because "the same three lines,
reordered" (notebook `0.`, Part 1) only covers *traversal order*. Once the question changes shape - "compute
something from children" vs. "enumerate every root-to-leaf path" - the *skeleton* of the recursion changes
too. Two skeletons cover a large share of tree DFS questions:

| Template | Shape | Answers questions like |
|---|---|---|
| **B.1 Return-a-value** | `if base case: return X`; combine children's return values (or pass state down) | height, balance, diameter, does-a-path-exist |
| **B.2 Path backtracking** | `choose a child -> recurse -> un-choose`, recording at each leaf | every root-to-leaf path, path sums, numbers/strings built along a path |


## B.1 - Return-a-value DFS

**Baseline - LC 104: Maximum Depth of Binary Tree.** The shape: handle the base case, recurse into both
children, **combine** their results into this node's answer.


In [ ]:
def max_depth(node):
    if not node:
        return 0
    return 1 + max(max_depth(node.left), max_depth(node.right))


t104 = build_tree([3, 9, 20, None, None, 15, 7])
print(max_depth(t104))             # expect 3
print(max_depth(None))             # expect 0


### B.1a - LC 110: Balanced Binary Tree

**Same as baseline, except:** instead of returning height alone, we **overload the return value** with a
sentinel: `-1` means "already found an imbalance somewhere below - stop bothering to compute further."
That single sentinel turns an `O(n^2)` "compute height at every node separately" solution into `O(n)` -
each subtree is measured exactly once, and a `-1` short-circuits every ancestor's check on the way back up.


In [ ]:
def is_balanced(root):
    def height(node):
        if not node:
            return 0
        lh = height(node.left)
        if lh == -1:
            return -1                          # left subtree already unbalanced - bail out early
        rh = height(node.right)
        if rh == -1:
            return -1
        if abs(lh - rh) > 1:
            return -1                          # THIS node is the one that's unbalanced
        return 1 + max(lh, rh)                 # otherwise behave exactly like max_depth
    return height(root) != -1


print(is_balanced(build_tree([3, 9, 20, None, None, 15, 7])))            # expect True
print(is_balanced(build_tree([1, 2, 2, 3, 3, None, None, 4, 4])))        # expect False


### B.1b - LC 543: Diameter of Binary Tree

**Same as baseline, except:** we still compute height exactly like `max_depth`, but along the way we stash
the **best `left_height + right_height` seen at any node** into a variable captured with `nonlocal`. The
diameter (longest path between any two nodes) doesn't have to pass through the root, so it can't be the
function's *return value* - it has to be tracked as a side effect while the height recursion runs anyway.


In [ ]:
def diameter_of_binary_tree(root):
    best = 0
    def height(node):
        nonlocal best
        if not node:
            return 0
        lh = height(node.left)
        rh = height(node.right)
        best = max(best, lh + rh)              # longest path THROUGH this node, checked at every node
        return 1 + max(lh, rh)                 # still just max_depth's return value
    height(root)
    return best


print(diameter_of_binary_tree(build_tree([1, 2, 3, 4, 5])))   # expect 3


### B.1c - LC 112: Path Sum

**Same as baseline, except:** instead of *returning* information up from children, we *pass* information
**down** to them (the remaining sum needed) - and the base case fires only at a true leaf, not at `None`.
Still the same "handle base case, recurse into children, combine" shape - just top-down instead of
bottom-up.


In [ ]:
def has_path_sum(root, target):
    if not root:
        return False
    if not root.left and not root.right:       # base case: a LEAF, not an empty subtree
        return root.val == target
    remaining = target - root.val               # pass state DOWN instead of combining state UP
    return has_path_sum(root.left, remaining) or has_path_sum(root.right, remaining)


t112 = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1])
print(has_path_sum(t112, 22))        # expect True  (5 -> 4 -> 11 -> 2)
print(has_path_sum(build_tree([1, 2]), 1))   # expect False


## B.2 - Path Backtracking DFS

**Baseline - LC 257: Binary Tree Paths.** The shape: **choose** a child, **recurse** into it with that
child appended to the running path, then **un-choose** it (pop) once that branch is fully explored. Record
the finished path only when a **leaf** is reached. This is the same choose/recurse/un-choose rhythm as
array backtracking - here the "candidates" at each step are just "this node's existing children" instead of
"the remaining items in a list."


In [ ]:
def binary_tree_paths(root):
    if not root:
        return []
    out, path = [], [str(root.val)]
    def backtrack(node):
        if not node.left and not node.right:
            out.append("->".join(path))         # RECORD - reached a leaf
            return
        for nxt in (node.left, node.right):
            if nxt:
                path.append(str(nxt.val))         # CHOOSE
                backtrack(nxt)                      # RECURSE
                path.pop()                          # UN-CHOOSE
    backtrack(root)
    return out


t257 = build_tree([1, 2, 3, None, 5])
print(binary_tree_paths(t257))     # expect ['1->2->5', '1->3']


### B.2a - LC 113: Path Sum II

**Same as baseline, except:** the path stores raw `int` values instead of strings (no need to join them),
and we track a `remaining` sum alongside the path. The "record" condition adds one clause - it's a leaf
**and** `remaining == 0` - otherwise the un-choose (`path.pop()`) still happens, exactly like the baseline.


In [ ]:
def path_sum_ii(root, target):
    out, path = [], []
    def backtrack(node, remaining):
        if not node:
            return
        path.append(node.val)                    # CHOOSE
        remaining -= node.val
        if not node.left and not node.right and remaining == 0:
            out.append(path[:])                    # RECORD - leaf AND target hit exactly
        else:
            backtrack(node.left, remaining)          # RECURSE
            backtrack(node.right, remaining)
        path.pop()                                # UN-CHOOSE (always, whether or not we recorded)
    backtrack(root, target)
    return out


t113 = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1])
print(path_sum_ii(t113, 22))       # expect [[5, 4, 11, 2], [5, 8, 4, 5]]


### B.2b - LC 129: Sum Root to Leaf Numbers

**Same as baseline, except:** we don't keep a `path` list at all - the running path is compressed into a
single number (`current = current * 10 + node.val`) that lives on the call stack as a plain function
argument, so there's nothing to explicitly "un-choose": each recursive call gets its own `current`, and it
simply evaporates when that call returns. Recording (`total += current`) happens at every leaf, accumulated
into a `nonlocal` total - the same trick B.1b used for diameter.


In [ ]:
def sum_root_to_leaf(root):
    total = 0
    def backtrack(node, current):
        nonlocal total
        if not node:
            return
        current = current * 10 + node.val         # CHOOSE - folded into the running number
        if not node.left and not node.right:
            total += current                        # RECORD - reached a leaf
            return
        backtrack(node.left, current)                # RECURSE (current resets naturally per call - no explicit pop)
        backtrack(node.right, current)
    backtrack(root, 0)
    return total


print(sum_root_to_leaf(build_tree([1, 2, 3])))     # expect 25  (12 + 13)


### B.2c - LC 988: Smallest String Starting From Leaf

**Same as baseline, except:** two small twists. First, the path is built **root-to-leaf** as usual but read
out **reversed** at each leaf (the problem wants leaf-to-root strings). Second, "record" doesn't just
append - it keeps a running best (`best[0]`), comparing the new candidate lexicographically. Same
choose/recurse/un-choose skeleton as `binary_tree_paths`, with the record step doing a comparison instead of
a plain append.


In [ ]:
def smallest_from_leaf(root):
    best = [None]                                # mutable box - simplest way to track "best so far" here
    path = []
    def backtrack(node):
        if not node:
            return
        path.append(chr(ord("a") + node.val))     # CHOOSE
        if not node.left and not node.right:
            s = "".join(reversed(path))             # RECORD - leaf-to-root, so reverse the root-to-leaf path
            if best[0] is None or s < best[0]:
                best[0] = s
        else:
            backtrack(node.left)                     # RECURSE
            backtrack(node.right)
        path.pop()                                  # UN-CHOOSE
    backtrack(root)
    return best[0]


t988 = build_tree([0, 1, 2, 3, 4, 3, 4])
print(smallest_from_leaf(t988))    # expect 'dba'


## B.3 - Recap: which template, and what changed

| Problem | Template | Same as baseline, except... |
|---|---|---|
| 104 Max Depth | B.1 return-a-value | *(this IS the baseline)* |
| 110 Balanced Tree | B.1 return-a-value | overload the return value with a `-1` sentinel to short-circuit |
| 543 Diameter | B.1 return-a-value | track a `nonlocal` best-so-far alongside the same height recursion |
| 112 Path Sum | B.1 return-a-value | pass state down (remaining sum) instead of combining state up |
| 257 Binary Tree Paths | B.2 path backtracking | *(this IS the baseline)* |
| 113 Path Sum II | B.2 path backtracking | track a numeric `remaining`; record only on leaf AND `remaining == 0` |
| 129 Sum Root to Leaf Numbers | B.2 path backtracking | fold the path into one number passed as an argument - no explicit pop needed |
| 988 Smallest String From Leaf | B.2 path backtracking | reverse the path at record time; keep a running "best so far" instead of collecting all paths |


## 🧩 Patterns Learned

- **BFS on a tree still collapses into one function**, `bfs_generic(sources, neighbors_fn,
  is_target_fn)`. Section A.2 shows the *only* thing that changes when a question needs to move in a
  direction a `TreeNode` doesn't give you for free (parent-ward): you hand-build the missing edge (a
  parent map), and the same nine-line BFS still solves it.
- **DFS on a tree has (at least) two distinct skeletons:**
  - **return-a-value** (`0.`'s bottom-up postorder, generalised - or threaded top-down as state) for
    anything computed from children;
  - **path backtracking** (`choose -> recurse -> un-choose`, record at leaves) for anything that
    enumerates root-to-leaf paths, sums, or strings.
- **The record step is where variants differ**, not the skeleton. `256`'s baseline just appends a path;
  every variant in B.2 keeps the exact same choose/recurse/un-choose shape and only changes *what counts
  as a valid record* (an exact sum, a running best, a folded number).
- **This is the same lesson as notebook `0.`'s sections 2.2->2.3 and 2.5, one level up:** own the
  *template*, and the *variant* is a one-or-two-line diff you can predict before you write it.
- **What's deliberately NOT here:** grid/graph problems (Number of Islands, Course Schedule, Word Ladder,
  Clone Graph, Rotting Oranges, ...) use these exact same templates - `bfs_generic` unchanged, and a
  visited-marking DFS skeleton very close to B.1 - but they're saved for a dedicated graph notebook so this
  one stays focused on trees.
